# Treinos simultâneos numa sessão

Roda vários treinos com o mesmo tamanho de imagem numa sessão só do Colab, ao mesmo tempo na mesma GPU. As imagens são processadas uma vez, cada treino roda num processo próprio, e a sessão desliga sozinha no fim (também se der erro). Antes da rodada, cada treino passa por uma execução de fumaça de poucos minutos.

Use GPU A100 com RAM alta. A lógica está em `src/retinopatia/execucao.py`. O andamento aparece aqui a cada 10 minutos, e o log de cada treino vai para `resultados/logs_paralelos/` no Drive.

## Preparação do repositório

In [ ]:
# O código do projeto vive no repositório do GitHub.
# Esta célula traz (ou atualiza) esse código dentro do Colab.

import os
import sys

URL_REPOSITORIO = "https://github.com/Rcorrea07/ML_Retinopatia.git"
PASTA_REPOSITORIO = "/content/ML_Retinopatia"

# Versão do código a usar. "origin/main" é a mais recente no GitHub.
# Para reproduzir um treino antigo, use a tag ou o commit dele
# (por exemplo "treino-v2"); o commit de cada treino fica no
# configuracao.json da pasta de resultados e no registro de experimentos.
REVISAO_CODIGO = "origin/main"

# Treinos desta sessão: (experimento, semente), todos com o mesmo
# tamanho de imagem (lista em src/retinopatia/experimentos.py). Rodam ao
# mesmo tempo na mesma GPU, quantos couberem na memória; os outros esperam.
EXECUCOES = [
    ("R10_b0_768", 17),
    ("R11_v2b0_768", 42),
    ("R11_v2b0_768", 17),
]

# O primeiro experimento define o tamanho das imagens preparadas
os.environ["RETINOPATIA_EXPERIMENTO"] = EXECUCOES[0][0]

# "1" = só as execuções de fumaça (poucos pacientes, uma época por fase)
os.environ["RETINOPATIA_FUMACA"] = "0"

# True = encerra a sessão no fim, também se der erro, para não gastar
# unidades com a GPU parada
DESLIGAR_AO_TERMINAR = True

if not os.path.exists(PASTA_REPOSITORIO):
    !git clone {URL_REPOSITORIO} {PASTA_REPOSITORIO}

!git -C {PASTA_REPOSITORIO} fetch --tags origin
!git -C {PASTA_REPOSITORIO} checkout --detach {REVISAO_CODIGO}
!git -C {PASTA_REPOSITORIO} log -1 --date=short --format="Código no commit %h (%cd): %s"

# Torna o pacote "retinopatia" importável neste notebook
caminho_src = os.path.join(PASTA_REPOSITORIO, "src")

if caminho_src not in sys.path:
    sys.path.insert(0, caminho_src)

# Recarrega os módulos automaticamente quando esta célula traz código
# novo, evitando ter que reiniciar o ambiente a cada edição no VS Code.
# O Colab usa Python 3.12+, que não tem mais o módulo "imp", mas a
# versão do IPython instalada nele ainda o importa no autoreload.
# Um substituto mínimo resolve; se mesmo assim falhar, o notebook
# segue sem autoreload (aí é preciso reiniciar após cada git pull).
import importlib
import types

try:
    import imp
except ModuleNotFoundError:
    sys.modules["imp"] = types.ModuleType("imp")
    sys.modules["imp"].reload = importlib.reload

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception as erro:
    print("Autoreload indisponível:", erro)

print("Repositório pronto em:", PASTA_REPOSITORIO)

## Conexão com o Drive

In [ ]:
from retinopatia import ambiente_colab, execucao

ambiente_colab.montar_drive()

## Rodada

Prepara as imagens, confere cada treino com uma fumaça, roda os treinos, consolida o registro e desliga a sessão.

In [ ]:
codigos = execucao.rodar_rodada(
    EXECUCOES,
    desligar_ao_terminar=DESLIGAR_AO_TERMINAR
)